<a href="https://colab.research.google.com/github/xsindgy/diamond-price-calculator/blob/main/diamond-price-calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# importando
import pandas as pd

In [15]:
# leitura de dados
df = pd.read_csv('/content/sample_data/diamonds.csv')

In [16]:
# adicionando inputs

print("=== Avaliação de Diamante ===")
print('''=== Os valores devem estar da seguinte maneira para que o filtro seja aplicado corretamente: ===

Peso em quilates do diamante (0,2 a 5,01)

Qualidade do corte (Regular, Bom, Muito Bom, Premium, Ideal)

Cor: Cor do diamante, de J (pior) a D (melhor)

Clareza: Medida da pureza do diamante (I1 (pior), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (melhor))\n''')

quilate = float(input("Digite o quilate: "))
corte = input("Digite o corte: ").strip().lower()
cor = input("Digite a cor: ").strip().lower()
pureza = input("Digite a pureza: ").strip().lower()

# Filtro exato
resultado = df[
    (df["carat"] == quilate) &
    (df["cut"].str.lower() == corte) &
    (df["color"].str.lower() == cor) &
    (df["clarity"].str.lower() == pureza)
]

# Se não encontrar, aplica margem progressiva
if resultado.empty:
    print("\nNenhum resultado exato. Buscando os mais próximos...\n")

    margem = 0.1
    encontrado = False

    while not encontrado and margem <= 1.0:
        resultado = df[
            (df["carat"].between(quilate - margem, quilate + margem)) &
            (df["cut"].str.lower() == corte) &
            (df["color"].str.lower() == cor) &
            (df["clarity"].str.lower() == pureza)
        ]

        if not resultado.empty:
            encontrado = True
            print(f"Resultados encontrados com margem de ±{margem} no quilate:\n")
        else:
            margem = round(margem + 0.1, 1)

    # Se ainda vazio, relaxa também corte/cor/pureza
    if resultado.empty:
        print("Expandindo busca para características similares...\n")
        resultado = df[
            (df["carat"].between(quilate - 0.5, quilate + 0.5))
        ]
        resultado = resultado.copy()
        resultado["similaridade"] = (
            (resultado["cut"].str.lower() == corte).astype(int) +
            (resultado["color"].str.lower() == cor).astype(int) +
            (resultado["clarity"].str.lower() == pureza).astype(int)
        )
        resultado = resultado.sort_values("similaridade", ascending=False).head(10)
        print("Top 10 resultados mais próximos:\n")


pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)
preco_medio = df['price'].mean()

print(resultado.to_string(index=False))


=== Avaliação de Diamante ===
=== Os valores devem estar da seguinte maneira para que o filtro seja aplicado corretamente: ===

Peso em quilates do diamante (0,2 a 5,01)

Qualidade do corte (Regular, Bom, Muito Bom, Premium, Ideal)

Cor: Cor do diamante, de J (pior) a D (melhor)

Clareza: Medida da pureza do diamante (I1 (pior), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (melhor))

Digite o quilate: 0.2
Digite o corte: Bom
Digite a cor: J
Digite a pureza: SI2

Nenhum resultado exato. Buscando os mais próximos...

Expandindo busca para características similares...

Top 10 resultados mais próximos:

O preço médio para diamantes com essas características específicas é: $3932.799721913237 (preço em dólares)
 Unnamed: 0  carat       cut color clarity  depth  table  price    x    y    z  similaridade
      40321   0.61 Very Good     J     SI2   58.1   63.0   1125 5.62 5.60 3.26             2
      36067   0.52     Ideal     J     SI2   61.6   55.0    924 5.16 5.19 3.19             2
      48331   0.